In [ ]:
import pandas as pd
import mysql.connector 

connection = mysql.connector.connect(
    host = "Localhost",
    user = "root",
    password = "12345",
    database="market_analysis"
)
cursor =  connection.cursor()
cursor


# Which customer segments have the highest response rate to campaigns?

In [2]:
high_res_q=""" CREATE VIEW highest_response AS
     SELECT Income_band, Age_band, Web_engagement_band, Family_band, Spend_segment, Customer_Count,
   
    ROUND(Campaign1_Accepted * 100.0 / Customer_Count, 2) AS Campaign1_Response_Rate,
    ROUND(Campaign2_Accepted * 100.0 / Customer_Count, 2) AS Campaign2_Response_Rate,
    ROUND(Campaign3_Accepted * 100.0 / Customer_Count, 2) AS Campaign3_Response_Rate,
    ROUND(Campaign4_Accepted * 100.0 / Customer_Count, 2) AS Campaign4_Response_Rate,
    ROUND(Campaign5_Accepted * 100.0 / Customer_Count, 2) AS Campaign5_Response_Rate,
    ROUND(Latest_Campaign_Accepted * 100.0 / Customer_Count, 2) AS Latest_Campaign_Response_Rate

FROM sql_segmentation_table

ORDER BY Campaign2_Response_Rate DESC limit 5;"""




# How do spending patterns across products (wine, fruits, meat, fish, sweets, gold) vary by age, income, marital status, and country?

In [3]:
Spend_pattern_q="""CREATE VIEW spending_pattern AS
    SELECT
    CASE
        WHEN m.Income < 28240.225 THEN 'Low Income'
        WHEN m.Income <= 86917.575 THEN 'Middle Income'
        ELSE 'High Income'
    END AS Income_band,
    
    CASE
        WHEN d.Age<45 THEN 'Young'
        When d.Age BETWEEN 45 and 63 THEN 'Middle aged'
        ELSE 'Senior'
    END AS Age_band,

    m.Marital_Status,

    m.Country,

    COUNT(*) AS Customer_Count,

    ROUND(AVG(m.MntWines), 2) AS Avg_Wine_Spend,
    ROUND(AVG(m.MntFruits), 2) AS Avg_Fruit_Spend,
    ROUND(AVG(m.MntMeatProducts), 2) AS Avg_Meat_Spend,
    ROUND(AVG(m.MntFishProducts), 2) AS Avg_Fish_Spend,
    ROUND(AVG(m.MntSweetProducts), 2) AS Avg_Sweets_Spend,
    ROUND(AVG(m.MntGoldProds), 2) AS Avg_Gold_Spend,

    ROUND(AVG(m.MntWines + m.MntFruits + m.MntMeatProducts + m.MntFishProducts + m.MntSweetProducts + m.MntGoldProds), 2) AS Avg_Total_Spend

    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID = d.ID

    GROUP BY
        Income_Band,
        Age_Band,
        m.Marital_Status,
        m.Country

    ORDER BY
    Avg_Total_Spend DESC;"""


# Which channels (web, store, catalog, deals) are most used by high-value customers, and how often do they visit the website?

In [6]:
Channel_q="""CREATE VIEW channel_based_highvalue AS
    SELECT
    CASE
        WHEN d.Total_Spend < 441 THEN 'Low Value'
        WHEN d.Total_Spend < 1600 THEN 'Medium Value'
        ELSE 'High Value'
    END AS Customer_Value,

    COUNT(*) AS Customer_Count,

    ROUND(AVG(m.NumWebPurchases), 2) AS Avg_Web_Purchases,
    ROUND(AVG(m.NumStorePurchases), 2) AS Avg_Store_Purchases,
    ROUND(AVG(m.NumCatalogPurchases), 2) AS Avg_Catalog_Purchases,
    ROUND(AVG(m.NumDealsPurchases), 2) AS Avg_Deals_Purchases,
    ROUND(AVG(m.NumWebVisitsMonth), 2) AS Avg_Website_Visits

    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID = d.ID

    GROUP BY Customer_Value

    ORDER BY
    CASE Customer_Value
        WHEN 'High Value' THEN 1
        WHEN 'Medium Value' THEN 2
        ELSE 3
    END;"""


# Are there specific segments that are under-served (low spending, high visits, low response)?

In [5]:
Under_served_q="""CREATE VIEW under_served AS
  SELECT Income_band, Age_band, Web_engagement_band, Family_band, Spend_segment, Customer_Count,

    ROUND((Campaign1_Accepted + Campaign2_Accepted + Campaign3_Accepted + Campaign4_Accepted + Campaign5_Accepted +
            Latest_Campaign_Accepted) * 100.0 / (Customer_Count * 6), 2) AS Overall_Response_Rate

FROM sql_segmentation_table

WHERE Spend_segment = 'Low spender'
  AND Web_engagement_band = 'High engagement'

ORDER BY Overall_Response_Rate ASC;"""


# What are characteristics of “ideal target customers” for future campaigns? (age range, income band, family composition, country, etc.)

In [4]:
ideal_target_q="""CREATE VIEW ideal_target AS
    SELECT
    CASE
        WHEN m.Income < 28240.225 THEN 'Low Income'
        WHEN m.Income <= 86917.575 THEN 'Middle Income'
        ELSE 'High Income'
    END AS Income_Band,

    CASE
        WHEN d.Age < 45 THEN 'Young'
        WHEN d.Age <= 63 THEN 'Middle-aged'
        ELSE 'Senior'
    END AS Age_Band,

    CASE
        WHEN m.Kidhome >= 1 AND m.Teenhome >= 1 THEN 'Parent of child & teen'
        WHEN m.Kidhome >= 1 THEN 'Parent of a child'
        WHEN m.Teenhome >= 1 THEN 'Parent of a teen'
        ELSE 'Adult'
    END AS Family_Band,

    m.Marital_Status,

    m.Country,

    COUNT(*) AS Customer_Count,

    ROUND((SUM(m.AcceptedCmp1) + SUM(m.AcceptedCmp2) + SUM(m.AcceptedCmp3) + SUM(m.AcceptedCmp4) + SUM(m.AcceptedCmp5) + 
        SUM(m.Response)) * 100.0 / (COUNT(*) * 6), 2) AS Overall_Response_Rate,

    ROUND(AVG(d.Total_Spend), 2) AS Avg_Total_Spend

    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID = d.ID

    GROUP BY
        Income_Band,
        Age_Band,
        Family_Band,
        m.Marital_Status,
        m.Country

    HAVING COUNT(*) >= 10

    ORDER BY
        Overall_Response_Rate DESC,
        Avg_Total_Spend DESC;"""


In [7]:
queries = [high_res_q, Spend_pattern_q, Channel_q, Under_served_q, ideal_target_q]


with open("SQL_bussiness_case.sql", "w", encoding="utf-8") as f:
    for q in queries:
        f.write(q)
        f.write("\n\n-- ==================================\n\n")

print("SQL_bussiness_case.sql created successfully!")

SQL_bussiness_case.sql created successfully!


# Income Segmentation

In [ ]:
income_seg_query = """CREATE VIEW income_segmentation AS 
    SELECT 
    CASE
        WHEN Income < 28240.225 THEN 'Low Income'
        WHEN Income <= 86917.575 THEN 'Middle Income'
        ELSE 'High Income'
    END AS Income_band,
    Count(*) AS Customer_Count
    FROM marketing_campaigns_data 
    Group BY Income_band;"""


# Age Segmentation

In [9]:
Age_seg_query="""CREATE VIEW age_segmentation AS
    SELECT
    CASE
        WHEN Age<45 THEN 'Young'
        When Age BETWEEN 45 and 63 THEN 'Middle aged'
        ELSE 'Senior'
        END AS Age_band,
        COUNT(*) AS Customer_COUNT
        FROM derived_table
        GROUP BY Age_band;"""


# Campaign Segmentation

In [10]:
# Accepted atleast one campaign
atleat_1campaign_query="""CREATE VIEW atleast_1campaign AS
    SELECT
    CASE 
        WHEN AcceptedCmp1 = 1
          OR AcceptedCmp2 = 1
          OR AcceptedCmp3 = 1
          OR AcceptedCmp4 = 1
          OR AcceptedCmp5 = 1
          OR Response = 1
        THEN 'Yes'
        ELSE 'No'
        END AS Accepted_campaign,
        COUNT(*) AS Customer_count
        From marketing_campaigns_data
        GROUP BY Accepted_campaign;"""


In [11]:
campaignwise_segmen_query="""CREATE VIEW campaignwise_segmentation AS
    SELECT
    'Campaign 1' AS Campaign,
    COUNT(CASE WHEN AcceptedCmp1 = 1 THEN 1 END) AS Accepted
FROM marketing_campaigns_data

UNION ALL

SELECT
    'Campaign 2',
    COUNT(CASE WHEN AcceptedCmp2 = 1 THEN 1 END)
FROM marketing_campaigns_data

UNION ALL

SELECT
    'Campaign 3',
    COUNT(CASE WHEN AcceptedCmp3 = 1 THEN 1 END)
FROM marketing_campaigns_data

UNION ALL

SELECT
    'Campaign 4',
    COUNT(CASE WHEN AcceptedCmp4 = 1 THEN 1 END)
FROM marketing_campaigns_data

UNION ALL

SELECT
    'Campaign 5',
    COUNT(CASE WHEN AcceptedCmp5 = 1 THEN 1 END)
FROM marketing_campaigns_data

UNION ALL

SELECT
    'Latest Campaign',
    COUNT(CASE WHEN Response = 1 THEN 1 END)
FROM marketing_campaigns_data;"""


# Web Engagement Segmentation

In [12]:
web_engage_query="""CREATE VIEW web_engagement AS
SELECT
    CASE
        WHEN NumWebVisitsMonth > 7 THEN 'High engagement'
        WHEN NumWebVisitsMonth Between 3 AND 7 THEN 'Average engagement'
        ELSE 'Low engagement'
        END AS Web_engagement_band,
        COUNT(*) AS Customer_count
        From marketing_campaigns_data
        GROUP BY Web_engagement_band;"""


# Family segmentation

In [14]:
family_seg_query="""CREATE VIEW family_segmentation AS
    SELECT
    CASE
        WHEN kidhome>=1 THEN 'Parent of a child'
        WHEN teenhome>=1 THEN 'Parent of a teen'
        ELSE 'Adult'
        END AS Family_band,
        COUNT(*) AS Customer_count
        FROM marketing_campaigns_data
        GROUP BY Family_band;"""


# Spender segmentation

In [15]:
spender_seg_query="""CREATE VIEW spender_segmentation AS
    SELECT
    CASE
        WHEN Total_Spend < 441 THEN 'Low spender'
        WHEN Total_Spend < 1600 THEN 'Medium spender'
        ELSE 'High spender'
    END AS Spend_segment,
    COUNT(*) AS Customer_Count
    FROM derived_table
    GROUP BY Spend_segment;"""


# Combined segmentation

In [16]:
combined_seg_query="""CREATE VIEW overall_segmentation AS
    SELECT 
    CASE
        WHEN m.Income < 28240.225 THEN 'Low Income'
        WHEN m.Income <= 86917.575 THEN 'Middle Income'
        ELSE 'High Income'
    END AS Income_band,
    
    CASE
        WHEN d.Age<45 THEN 'Young'
        When d.Age BETWEEN 45 and 63 THEN 'Middle aged'
        ELSE 'Senior'
    END AS Age_band,
    
    CASE
        WHEN m.NumWebVisitsMonth > 7 THEN 'High engagement'
        WHEN m.NumWebVisitsMonth Between 3 AND 7 THEN 'Average engagement'
        ELSE 'Low engagement'
    END AS Web_engagement_band,
    
    CASE
        WHEN m.Kidhome >= 1 AND m.Teenhome >= 1 THEN 'Parent of child & teen'
        WHEN m.kidhome>=1 THEN 'Parent of a child'
        WHEN m.teenhome>=1 THEN 'Parent of a teen'
        ELSE 'Adult'
    END AS Family_band,
    
    CASE
        WHEN d.Total_Spend < 441 THEN 'Low spender'
        WHEN d.Total_Spend < 1600 THEN 'Medium spender'
        ELSE 'High spender'
    END AS Spend_segment,
    
    COUNT(*) AS Customer_Count,

    SUM(CASE WHEN m.AcceptedCmp1 = 1 THEN 1 ELSE 0 END) AS Campaign1_Accepted,
    SUM(CASE WHEN m.AcceptedCmp2 = 1 THEN 1 ELSE 0 END) AS Campaign2_Accepted,
    SUM(CASE WHEN m.AcceptedCmp3 = 1 THEN 1 ELSE 0 END) AS Campaign3_Accepted,
    SUM(CASE WHEN m.AcceptedCmp4 = 1 THEN 1 ELSE 0 END) AS Campaign4_Accepted,
    SUM(CASE WHEN m.AcceptedCmp5 = 1 THEN 1 ELSE 0 END) AS Campaign5_Accepted,
    SUM(CASE WHEN m.Response = 1 THEN 1 ELSE 0 END) AS Latest_Campaign_Accepted

    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID = d.ID
    Group BY Income_band, Age_band, Web_engagement_band, Family_band, Spend_segment

    ORDER BY Customer_count DESC;"""


In [18]:
seg_queries = [income_seg_query, Age_seg_query, atleat_1campaign_query, campaignwise_segmen_query, web_engage_query, family_seg_query, spender_seg_query, spender_seg_query, combined_seg_query]


with open("Segmentation.sql", "w", encoding="utf-8") as f:
    for q in seg_queries:
        f.write(q)
        f.write("\n\n-- ==================================\n\n")

print("Segmentation.sql created successfully!")

Segmentation.sql created successfully!


# Overall Campaign Acceptance Rate

In [ ]:
KPI_campaign_acceptance_query="""CREATE VIEW KPI_campacceptance AS
    SELECT 
    ROUND((SUM(AcceptedCmp1)+SUM(AcceptedCmp2)+SUM(AcceptedCmp3)+SUM(AcceptedCmp4)+SUM(AcceptedCmp5)+SUM(Response))*100.0/(COUNT(*)*6), 2) AS OVERALL_response_rate
    FROM marketing_campaigns_data;"""
   

# Average spend

In [ ]:
KPI_avg_spend_query="""CREATE VIEW KPI_avgspend AS
SELECT ROUND(AVG(Total_Spend),2) AS AVG_customer_spend FROM derived_table;"""


# Average spend per segment

In [ ]:
KPI_avgspend_seg_query="""CREATE VIEW KPI_avgspend_seg AS
    SELECT
    ROUND(AVG(MntWines),2) AS Avg_on_wines,
    ROUND(AVG(MntFruits),2) AS Avg_on_fruits,
    ROUND(AVG(MntMeatProducts),2) AS Avg_on_meat,
    ROUND(AVG(MntFishProducts),2) AS Avg_on_fish,
    ROUND(AVG(MntSweetProducts),2) AS Avg_on_sweets,
    ROUND(AVG(MntGoldProds),2) AS Avg_on_gold
     
    FROM marketing_campaigns_data;"""


# Average store, web and catalog purchases

In [ ]:
KPI_avgpur_query="""CREATE VIEW KPI_avgpurchase AS
SELECT
    ROUND(AVG(NumStorePurchases),2) AS Avg_store_purchases,
    ROUND(AVG(NumWebPurchases),2) AS Avg_web_purchases,
    ROUND(AVG(NumCatalogPurchases),2) AS Avg_catalog_purchases
     
    FROM marketing_campaigns_data;"""


# Average website visits

In [ ]:
KPI_avgweb_query="""CREATE VIEW KPI_avg_webvisit AS
    SELECT
    ROUND(AVG(NumWebVisitsMonth),2) AS Avg_web_visit
    FROM marketing_campaigns_data;"""


# KPI by Education

In [ ]:
KPI_edu_query="""CREATE VIEW KPI_edu AS
SELECT m.Education, COUNT(*) AS Customers,
    ROUND(AVG(d.Total_Spend),2) AS Avg_Spend,
    ROUND(AVG(m.NumWebVisitsMonth),2) AS Avg_Web_Visits,
    ROUND((SUM(m.AcceptedCmp1)+SUM(m.AcceptedCmp2)+SUM(m.AcceptedCmp3)+SUM(m.AcceptedCmp4)+SUM(m.AcceptedCmp5)+SUM(m.Response))*100.0/(COUNT(*)*6), 2) AS Overall_response_rate
    
    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID=d.ID
    
    GROUP BY Education
    ORDER BY Avg_Spend DESC;"""


# KPI by Age band

In [ ]:
KPI_age_query="""CREATE VIEW KPI_age AS
SELECT 
    CASE
        WHEN Age BETWEEN 30 AND 44 THEN 'Young Adults'
        WHEN Age BETWEEN 45 AND 59 THEN 'Middle Aged'
        ELSE 'Senior Adults'
    END AS Age_band,
    COUNT(*) AS Customers,
    ROUND(AVG(d.Total_Spend),2) AS Avg_Spend,
    ROUND(AVG(m.NumWebVisitsMonth),2) AS Avg_Web_Visits,
    ROUND((SUM(m.AcceptedCmp1)+SUM(m.AcceptedCmp2)+SUM(m.AcceptedCmp3)+SUM(m.AcceptedCmp4)+SUM(m.AcceptedCmp5)+SUM(m.Response))*100.0/(COUNT(*)*6), 2) AS Overall_response_rate
    
    FROM marketing_campaigns_data m
    JOIN derived_table d ON m.ID=d.ID
     
    GROUP BY Age_band
    ORDER BY Avg_Spend DESC;"""


# KPI by marital status

In [ ]:
KPI_maritalstatus_query="""CREATE VIEW KPI_maritalstatus AS
    SELECT m.Marital_Status, COUNT(*) AS Customers,
    ROUND(AVG(d.Total_Spend),2) AS Avg_Spend,
    ROUND(AVG(m.NumWebVisitsMonth),2) AS Avg_Web_Visits,
    ROUND((SUM(m.AcceptedCmp1)+SUM(m.AcceptedCmp2)+SUM(m.AcceptedCmp3)+SUM(m.AcceptedCmp4)+SUM(m.AcceptedCmp5)+SUM(m.Response))*100.0/(COUNT(*)*6), 2) AS Overall_response_rate
    
    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID=d.ID
    
    GROUP BY m.Marital_Status
    ORDER BY Avg_Spend DESC;"""


# KPI by country

In [ ]:
KPI_country_query="""CREATE VIEW KPI_country AS
SELECT m.Country, COUNT(*) AS Customers,
    ROUND(AVG(d.Total_Spend),2) AS Avg_Spend,
    ROUND(AVG(m.NumWebVisitsMonth),2) AS Avg_Web_Visits,
    ROUND((SUM(m.AcceptedCmp1)+SUM(m.AcceptedCmp2)+SUM(m.AcceptedCmp3)+SUM(m.AcceptedCmp4)+SUM(m.AcceptedCmp5)+SUM(m.Response))*100.0/(COUNT(*)*6), 2) AS Overall_response_rate
    
    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID=d.ID
    
    GROUP BY m.Country
    ORDER BY Avg_Spend DESC;"""


# KPI by Income_band

In [ ]:
KPI_incomeband_query="""CREATE VIEW KPI_incomeband AS
    SELECT 
    CASE
        WHEN m.Income < 28240.225 THEN 'Low Income'
        WHEN m.Income <= 86917.575 THEN 'Middle Income'
        ELSE 'High Income'
    END AS Income_band,
    COUNT(*) AS Customers,
    ROUND(AVG(d.Total_Spend),2) AS Avg_Spend,
    ROUND(AVG(m.NumWebVisitsMonth),2) AS Avg_Web_Visits,
    ROUND((SUM(m.AcceptedCmp1)+SUM(m.AcceptedCmp2)+SUM(m.AcceptedCmp3)+SUM(m.AcceptedCmp4)+SUM(m.AcceptedCmp5)+SUM(m.Response))*100.0/(COUNT(*)*6), 2) AS Overall_response_rate
    
    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID=d.ID
    
    GROUP BY Income_band
    ORDER BY Avg_Spend DESC;"""


# Segmentwise analysis

In [29]:
KPI_seg_query="""CREATE VIEW KPI_segmentation AS
    SELECT 
    CASE
        WHEN Age BETWEEN 30 AND 44 THEN 'Young Adults'
        WHEN Age BETWEEN 45 AND 59 THEN 'Middle Aged'
        ELSE 'Senior Adults'
    END AS Age_band,
    CASE
        WHEN m.Income < 28240.225 THEN 'Low Income'
        WHEN m.Income <= 86917.575 THEN 'Middle Income'
        ELSE 'High Income'
    END AS Income_band,
    COUNT(*) AS Customers,
    ROUND(AVG(d.Total_Spend),2) AS Avg_Spend,
    ROUND(AVG(m.NumWebVisitsMonth),2) AS Avg_Web_Visits,
    ROUND(AVG(m.MntWines),2) AS Avg_on_wines,
    ROUND(AVG(m.MntFruits),2) AS Avg_on_fruits,
    ROUND(AVG(m.MntMeatProducts),2) AS Avg_on_meat,
    ROUND(AVG(m.MntFishProducts),2) AS Avg_on_fish,
    ROUND(AVG(m.MntSweetProducts),2) AS Avg_on_sweets,
    ROUND(AVG(m.MntGoldProds),2) AS Avg_on_gold,
    ROUND(AVG(m.NumDealsPurchases),2) AS Avg_new_deal,
    ROUND(AVG(m.NumWebPurchases),2) AS Avg_web_pur,
    ROUND(AVG(m.NumCatalogPurchases),2) AS Avg_catalog,
    ROUND(AVG(m.NumStorePurchases),2) AS Avg_store_pur,

    ROUND(AVG(m.AcceptedCmp1) * 100, 2) AS Campaign1_Acceptance_Rate,
    ROUND(AVG(m.AcceptedCmp2) * 100, 2) AS Campaign2_Acceptance_Rate,
    ROUND(AVG(m.AcceptedCmp3) * 100, 2) AS Campaign3_Acceptance_Rate,
    ROUND(AVG(m.AcceptedCmp4) * 100, 2) AS Campaign4_Acceptance_Rate,
    ROUND(AVG(m.AcceptedCmp5) * 100, 2) AS Campaign5_Acceptance_Rate,
    ROUND(AVG(m.Response) * 100, 2) AS Last_Campaign_Response_Rate
    
    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID=d.ID
    
    GROUP BY Income_band, Age_band
    ORDER BY Avg_Spend DESC;"""


In [31]:
KPI_queries=[KPI_campaign_acceptance_query, KPI_avg_spend_query, KPI_avgspend_seg_query, KPI_avgpur_query, KPI_avgweb_query, KPI_edu_query, KPI_age_query, KPI_maritalstatus_query, KPI_country_query, KPI_incomeband_query, KPI_seg_query]
with open("KPI.sql", "w", encoding="utf-8") as f:
    for q in KPI_queries:
        f.write(q)
        f.write("\n\n-- ==================================\n\n")

print("KPI.sql created successfully!")

KPI.sql created successfully!
